# Notebook 02 — Preprocessing Pipeline V2

**Goal:** Transform raw CSV → clean, split, scaled artifacts ready for modeling.

**V2 Changes from V1:**
- 21 features instead of 14 (added timing, velocity, biotech, region signals)
- Log-transforms applied during feature engineering (not in the sklearn pipeline)
- Also saves **3-class** (no operating) and **binary** (success/failed) dataset variants
- Saves feature group definitions for ablation studies
- SMOTE saved separately (modeling notebooks decide whether to use it)

**Inputs:** `big_startup_secsees_dataset.csv`
**Outputs:** `artifacts/` folder with all splits, encoders, and metadata


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import joblib
import yaml

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

# Import V2 transformers
from utils.feature_engineering import (
    RawCleaner, FeatureEngineerV2, CategoryGrouper, FeatureSelector,
    build_preprocessing_pipeline, get_feature_names,
    ALL_FEATURE_COLS, NUM_COLS, CAT_COLS, BINARY_COLS,
    FEATURE_GROUPS, VALID_STATUSES,
)

# Load config
with open('../configs/params.yaml', encoding='utf-8') as f:
    CFG = yaml.safe_load(f)

RANDOM_SEED = CFG['random_seed']
np.random.seed(RANDOM_SEED)
os.makedirs('artifacts', exist_ok=True)

print('All imports loaded. Pipeline V2 ready.')
print(f'Expected features: {len(ALL_FEATURE_COLS)} ({len(NUM_COLS)} num + {len(CAT_COLS)} cat + {len(BINARY_COLS)} bin)')


## 1. Load & Clean Raw Data

In [ ]:
# Load raw CSV
df_raw = pd.read_csv(CFG['paths']['raw_data'], skipinitialspace=True)
df_raw.columns = df_raw.columns.str.strip()
print(f'Raw dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')

# Clean with RawCleaner
cleaner = RawCleaner()
df_clean = cleaner.fit_transform(df_raw)

# Filter to valid statuses only
df_clean = df_clean[df_clean['status'].isin(VALID_STATUSES)].reset_index(drop=True)

print(f'After cleaning: {df_clean.shape[0]:,} rows')
print()
print('Status distribution:')
print(df_clean['status'].value_counts())
print()
print(df_clean['status'].value_counts(normalize=True).map('{:.1%}'.format))


## 2. Feature Engineering V2

In [ ]:
# Apply V2 feature engineering (row-level, no fitting needed)
engineer = FeatureEngineerV2()
df_features = engineer.fit_transform(df_clean)

new_cols = [c for c in df_features.columns if c not in df_clean.columns]
print(f'After feature engineering: {df_features.shape[1]} columns')
print(f'New features created ({len(new_cols)}): {new_cols}')


## 3. Category Grouping

In [ ]:
grouper = CategoryGrouper(
    top_n_countries=CFG['features']['top_n_countries'],
    top_n_categories=CFG['features']['top_n_categories'],
    top_n_regions=CFG['features'].get('top_n_regions', 15),
)

# Fit on full data (safe: only learning label lists, not statistics)
grouper.fit(df_features)

print(f'Top countries ({len(grouper.top_countries_)}): {grouper.top_countries_}')
print(f'Top categories ({len(grouper.top_categories_)}): {grouper.top_categories_[:5]}...')
print(f'Top regions ({len(grouper.top_regions_)}): {grouper.top_regions_[:5]}...')

df_grouped = grouper.transform(df_features)


## 4. Select Features & Encode Target

In [ ]:
# Select only the columns the pipeline expects
selector = FeatureSelector(columns=ALL_FEATURE_COLS)
X_raw = selector.transform(df_grouped)

print(f'Feature matrix: {X_raw.shape}')
print(f'Columns ({len(X_raw.columns)}): {X_raw.columns.tolist()}')
print()
# Verify no unexpected NaN in binary columns
for col in BINARY_COLS:
    nan_pct = X_raw[col].isna().mean()
    if nan_pct > 0:
        print(f'  WARNING: {col} has {nan_pct:.1%} NaN')
    else:
        print(f'  ✓ {col}: no NaN')


In [ ]:
# Encode the 4-class target
le_target = LabelEncoder()
y_full = le_target.fit_transform(df_clean['status'])
CLASS_NAMES = le_target.classes_.tolist()

print(f'Class mapping: {dict(zip(CLASS_NAMES, le_target.transform(CLASS_NAMES)))}')
print(f'y shape: {y_full.shape}, dtype: {y_full.dtype}')


## 5. Stratified Train / Val / Test Split

In [ ]:
# 70% train, 15% val, 15% test
X_train_raw, X_temp, y_train, y_temp = train_test_split(
    X_raw, y_full,
    test_size=CFG['split']['test_size'],
    random_state=RANDOM_SEED,
    stratify=y_full,
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=CFG['split']['val_ratio'],
    random_state=RANDOM_SEED,
    stratify=y_temp,
)

for name, X_s, y_s in [('Train', X_train_raw, y_train),
                        ('Val', X_val_raw, y_val),
                        ('Test', X_test_raw, y_test)]:
    pct = len(X_s) / len(X_raw) * 100
    dist = dict(zip(le_target.classes_, np.bincount(y_s, minlength=len(CLASS_NAMES))))
    print(f'{name:5s}: {len(X_s):,} rows ({pct:.0f}%) | {dist}')


## 6. Build & Fit Preprocessing Pipeline (train only)

In [ ]:
# Build the V2 preprocessing pipeline
preprocessor = build_preprocessing_pipeline()

# FIT ON TRAINING DATA ONLY
preprocessor.fit(X_train_raw)

# Leakage verification
scaler = preprocessor.named_transformers_['num'].named_steps['scaler']
assert hasattr(scaler, 'center_'), 'DATA LEAKAGE: scaler not fitted!'
print(f'✓ Scaler fitted with {len(scaler.center_)} features (no leakage)')

## 7. Transform All Splits

In [ ]:
# Apply the SAME learned transformation to all splits
X_train = preprocessor.transform(X_train_raw)
X_val   = preprocessor.transform(X_val_raw)
X_test  = preprocessor.transform(X_test_raw)

feature_names = get_feature_names()

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')
print(f'Features ({len(feature_names)}): {feature_names}')


## 8. Build Dataset Variants

We create 3-class and binary variants for modeling notebooks.
The modeling notebooks will decide which variant to use — nb02 just prepares them.


In [ ]:
# ── 3-class: drop "operating" ──
op_label = le_target.transform(['operating'])[0]

mask_train_3c = y_train != op_label
mask_val_3c = y_val != op_label
mask_test_3c = y_test != op_label

X_train_3c = X_train[mask_train_3c]
X_val_3c = X_val[mask_val_3c]
X_test_3c = X_test[mask_test_3c]

# Re-encode to 0,1,2
le_3class = LabelEncoder()
y_train_3c = le_3class.fit_transform(le_target.inverse_transform(y_train[mask_train_3c]))
y_val_3c = le_3class.transform(le_target.inverse_transform(y_val[mask_val_3c]))
y_test_3c = le_3class.transform(le_target.inverse_transform(y_test[mask_test_3c]))

print('3-CLASS VARIANT (closed / acquired / ipo):')
print(f'  Classes: {le_3class.classes_.tolist()}')
print(f'  Train: {X_train_3c.shape[0]:,}  Val: {X_val_3c.shape[0]:,}  Test: {X_test_3c.shape[0]:,}')
print(f'  Train dist: {dict(zip(le_3class.classes_, np.bincount(y_train_3c)))}')


In [ ]:
# ── Binary: success (acquired+ipo) vs failed (closed) ──
def make_binary(y_encoded, le):
    labels = le.inverse_transform(y_encoded)
    mask = np.isin(labels, ['closed', 'acquired', 'ipo'])
    y_bin = np.where(np.isin(labels, ['acquired', 'ipo']), 1, 0)  # 1=success, 0=failed
    return mask, y_bin

mask_train_bin, y_train_bin_full = make_binary(y_train, le_target)
mask_val_bin, y_val_bin_full = make_binary(y_val, le_target)
mask_test_bin, y_test_bin_full = make_binary(y_test, le_target)

# Filter out operating
X_train_bin = X_train[mask_train_bin]
X_val_bin = X_val[mask_val_bin]
X_test_bin = X_test[mask_test_bin]
y_train_bin = y_train_bin_full[mask_train_bin]
y_val_bin = y_val_bin_full[mask_val_bin]
y_test_bin = y_test_bin_full[mask_test_bin]

le_binary = LabelEncoder()
le_binary.classes_ = np.array(['failed', 'success'])

print('BINARY VARIANT (failed=0 / success=1):')
print(f'  Train: {X_train_bin.shape[0]:,}  Val: {X_val_bin.shape[0]:,}  Test: {X_test_bin.shape[0]:,}')
print(f'  Train dist: failed={np.sum(y_train_bin==0):,}  success={np.sum(y_train_bin==1):,}')


## 9. SMOTE (training set only)

In [ ]:
print('4-CLASS — Class distribution BEFORE SMOTE:')
for i, name in enumerate(CLASS_NAMES):
    cnt = np.sum(y_train == i)
    print(f'  {name:12s} {cnt:,} ({cnt/len(y_train):.1%})')

sm = SMOTE(random_state=RANDOM_SEED, k_neighbors=CFG['smote']['k_neighbors'])

# 4-class SMOTE
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print(f'\nAfter SMOTE: {len(y_train):,} → {len(y_train_res):,}')

# 3-class SMOTE
X_train_3c_res, y_train_3c_res = sm.fit_resample(X_train_3c, y_train_3c)
print(f'3-class SMOTE: {len(y_train_3c):,} → {len(y_train_3c_res):,}')

# Binary doesn't need SMOTE (already ~balanced: 53% vs 47%)
print(f'Binary: no SMOTE needed (balance: {np.mean(y_train_bin):.1%} success)')


## 10. Save All Artifacts

In [ ]:
# ── Preprocessing objects ──
joblib.dump(preprocessor, '../artifacts/preprocessor.pkl')
joblib.dump(feature_names, '../artifacts/feature_names.pkl')
joblib.dump(cleaner, '../artifacts/cleaner.pkl')
joblib.dump(engineer, '../artifacts/engineer.pkl')
joblib.dump(grouper, '../artifacts/grouper.pkl')
joblib.dump(selector, '../artifacts/selector.pkl')
joblib.dump(le_target, '../artifacts/label_encoder_target.pkl')
joblib.dump(le_3class, '../artifacts/label_encoder_3class.pkl')
joblib.dump(le_binary, '../artifacts/label_encoder_binary.pkl')

# ── Top-N lists ──
joblib.dump(grouper.top_countries_, '../artifacts/top_countries.pkl')
joblib.dump(grouper.top_categories_, '../artifacts/top_categories.pkl')
joblib.dump(grouper.top_regions_, '../artifacts/top_regions.pkl')

# ── Feature groups (for ablation) ──
joblib.dump(FEATURE_GROUPS, '../artifacts/feature_groups.pkl')

# ── 4-CLASS splits ──
joblib.dump(X_train, '../artifacts/X_train.pkl')
joblib.dump(X_val,   '../artifacts/X_val.pkl')
joblib.dump(X_test,  '../artifacts/X_test.pkl')
joblib.dump(y_train, '../artifacts/y_train.pkl')
joblib.dump(y_val,   '../artifacts/y_val.pkl')
joblib.dump(y_test,  '../artifacts/y_test.pkl')

# ── 4-CLASS SMOTE ──
joblib.dump(X_train_res, '../artifacts/X_train_smote.pkl')
joblib.dump(y_train_res, '../artifacts/y_train_smote.pkl')

# ── 3-CLASS splits ──
joblib.dump(X_train_3c, '../artifacts/X_train_3c.pkl')
joblib.dump(X_val_3c,   '../artifacts/X_val_3c.pkl')
joblib.dump(X_test_3c,  '../artifacts/X_test_3c.pkl')
joblib.dump(y_train_3c, '../artifacts/y_train_3c.pkl')
joblib.dump(y_val_3c,   '../artifacts/y_val_3c.pkl')
joblib.dump(y_test_3c,  '../artifacts/y_test_3c.pkl')
joblib.dump(X_train_3c_res, '../artifacts/X_train_3c_smote.pkl')
joblib.dump(y_train_3c_res, '../artifacts/y_train_3c_smote.pkl')

# ── BINARY splits ──
joblib.dump(X_train_bin, '../artifacts/X_train_bin.pkl')
joblib.dump(X_val_bin,   '../artifacts/X_val_bin.pkl')
joblib.dump(X_test_bin,  '../artifacts/X_test_bin.pkl')
joblib.dump(y_train_bin, '../artifacts/y_train_bin.pkl')
joblib.dump(y_val_bin,   '../artifacts/y_val_bin.pkl')
joblib.dump(y_test_bin,  '../artifacts/y_test_bin.pkl')

print('All artifacts saved to artifacts/')
print()
for f in sorted(os.listdir('../artifacts')):
    size = os.path.getsize(f'../artifacts/{f}') / 1024
    print(f'  {f:45s} {size:8.1f} KB')


## 11. Sanity Checks

In [ ]:
print('=== Sanity Checks ===')
print()

# 1. No NaN
for name, arr in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test),
                  ('X_train_3c', X_train_3c), ('X_train_bin', X_train_bin)]:
    nan_count = np.isnan(arr).sum()
    status = '✓' if nan_count == 0 else '✗ FIX THIS'
    print(f'{name:15s} NaN: {nan_count}  {status}')

# 2. Feature count matches
for name, arr in [('4-class', X_train), ('3-class', X_train_3c), ('binary', X_train_bin)]:
    assert arr.shape[1] == len(feature_names), f'{name} feature mismatch!'
    print(f'{name:15s} features: {arr.shape[1]} = {len(feature_names)} ✓')

# 3. No leakage: val/test untouched by SMOTE
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)
print(f'\nVal  size: {len(X_val):,} ✓ (untouched)')
print(f'Test size: {len(X_test):,} ✓ (untouched)')

# 4. 3-class has no operating
assert op_label not in y_train_3c
assert op_label not in y_val_3c
print(f'3-class: no operating label ✓')

# 5. Binary is 0/1 only
assert set(np.unique(y_train_bin)) == {0, 1}
print(f'Binary: labels are {{0, 1}} ✓')

print()
print('All sanity checks passed. Artifacts ready for nb03 (ML) and nb04 (DL).')
